# 04 - Baseline Model

Trains a baseline XGBoost model and a logistic regression for comparison, both with
genuinely default, unweighted settings -- no imbalance handling yet (that's Phase 5).
This notebook only touches train and validation. Test stays untouched until Phase 5's
final, one-time check.

## Why not just use accuracy?

Laundering is ~0.098% of transactions (Phase 2). A model that predicts "normal" for
every single transaction scores ~99.9% accuracy while catching zero laundering.
Accuracy can't tell that useless model apart from a genuinely good one -- it's
dominated by the huge number of easy "normal" predictions. Every metric used below is
chosen specifically to look at how the model handles the rare positive class instead.

## The metrics

- **Precision**: of everything flagged as laundering, what fraction actually was?
- **Recall**: of everything that actually was laundering, what fraction did we catch?
- **F1**: balances precision and recall into one number
- **ROC-AUC**: how well the model ranks a random laundering transaction above a random
  normal one. Reported for reference, but NOT the main metric -- with 99.9% of the data
  being "normal", it's easy to rank the huge number of obvious true negatives correctly,
  so this can look deceptively good even for a mediocre model.
- **PR-AUC (average precision)** -- the PRIMARY metric. Focuses specifically on the
  positive class, not inflated by easy true negatives the way ROC-AUC is.
- **Recall at a fixed 1% false-positive rate**: "if analysts can only tolerate reviewing
  1% of all normal transactions as false alarms, how much real laundering do we catch?"
- **Business cost**: `(missed laundering x $5,000) + (false alerts x $25)`, using the
  ASSUMED figures in `config.yaml` -- not real bank numbers, labelled as such everywhere.
  The decision threshold (the cutoff turning a probability into a yes/no flag) is chosen
  to minimise this cost, using ONLY the validation set.

In [ ]:
import sys
import os
from pathlib import Path

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.config import load_config
from src.train import (
    load_feature_tables, train_xgboost, prepare_xgb_data,
    train_logistic_regression, prepare_logreg_input, TARGET_COL,
)
from src.evaluate import evaluate_model, save_metrics

config = load_config(project_root / "config.yaml")
FIGURES_DIR = Path(config["paths"]["figures_dir"])
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

COLOR_XGB = "#2a78d6"
COLOR_LOGREG = "#eb6834"

## Load train and validation only

Test is deliberately never loaded in this notebook.

In [ ]:
train_df, val_df = load_feature_tables(config)
y_val = val_df[TARGET_COL]

print(f"train: {len(train_df):,} rows ({train_df[TARGET_COL].mean():.4%} laundering)")
print(f"val:   {len(val_df):,} rows ({y_val.mean():.4%} laundering)")

## Train XGBoost (default settings)

In [ ]:
xgb_model = train_xgboost(train_df, val_df, config)
xgb_scores = xgb_model.predict_proba(prepare_xgb_data(val_df))[:, 1]
xgb_metrics = evaluate_model(y_val, xgb_scores, config)
xgb_metrics

## Train logistic regression (default settings)

In [ ]:
logreg_model, preprocessor = train_logistic_regression(train_df, config)
X_val_logreg = preprocessor.transform(prepare_logreg_input(val_df))
logreg_scores = logreg_model.predict_proba(X_val_logreg)[:, 1]
logreg_metrics = evaluate_model(y_val, logreg_scores, config)
logreg_metrics

## Compare

Both models are trained with plain, unweighted default settings -- no scale_pos_weight,
no class_weight. Expect both to struggle somewhat here, especially on precision/recall
tradeoffs: this is the honest "before" picture. Phase 5 investigates whether
scale_pos_weight, undersampling, or hyperparameter tuning actually fixes this.

In [ ]:
comparison = pd.DataFrame({
    "xgboost_default": xgb_metrics,
    "logistic_regression": logreg_metrics,
}).T
comparison

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
metrics_to_plot = ["pr_auc", "roc_auc", "recall_at_fixed_fpr", "precision", "recall", "f1"]
x = np.arange(len(metrics_to_plot))
width = 0.35

ax.bar(x - width/2, [xgb_metrics[m] for m in metrics_to_plot], width, label="XGBoost (default)", color=COLOR_XGB)
ax.bar(x + width/2, [logreg_metrics[m] for m in metrics_to_plot], width, label="Logistic regression", color=COLOR_LOGREG)
ax.set_xticks(x)
ax.set_xticklabels(metrics_to_plot, rotation=30, ha="right")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.legend(frameon=False)
ax.set_title("Baseline model comparison on validation (default settings, no imbalance handling)")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "09_baseline_model_comparison.png", dpi=150)
plt.show()

## Save metrics

In [ ]:
results = {"xgboost_default": xgb_metrics, "logistic_regression": logreg_metrics}
save_metrics(results, config)
print(f"Saved to {Path(config['paths']['metrics_dir']) / 'baseline.json'}")